#### Memory and Chat History Management

In [1]:
## first we need to understand the problem 
# suppose we have simple chatmodel 
import os 
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq
llm = ChatGroq(model = os.getenv("groq_model_name"))
response = llm.invoke(
    "My name is Subbu"
)
print(response.content) # here it will give something greeting right

c:\Users\subramani.v\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Nice to meet you, Subbu! Is there something I can help you with or would you like to chat?


In [3]:
# now ask the llm to give me your name 
response = llm.invoke("what is my name?")
print(response.content)

I don't have any information about your name as we are starting a new conversation. I'm a large language model, I don't have personal knowledge or retain information about individual users. Each time you interact with me, it's a new conversation and I don't retain any information from previous conversations. If you'd like to share your name with me, I can use it to address you in our conversation.


- see here model will say i dont know your name. 
- why ? : becuase Every LLm call is independent. 
- the model only sees ```What is my name?```, it never saw ```my name is subbu```


### Interview Concept 
- LLms do not have memory by defualt.
- every request is stateless 
- think like below 
```
Request 1
    ↓
LLM
    ↓
Response

Request 2
    ↓
LLM
    ↓
Response
```

- The model forgets everything after the response

#### Then How does ChatGpt remembers ?
- The application sends previous messages again 
- insetad of ``` what is my name ``` 
- application sends :
```
Human: My name is Subbu

AI: Nice to meet you Subbu.

Human: What is my name?
```

- now the model can answer , becuase here the application is storing the memory not the model , so here simply previous responses are storing in memory and passing to the model so that it will answer.

####  This is Memory
- memory simply means 
```
Store Previous Messages
           ↓
Send Them Again
           ↓
Model Gets Context
```
- the model itself remembers nothing.
- the application remembers.
- This is the most important concpet.

#### Visual flow 
- without memory 
```
User
 ↓
Question
 ↓
LLM
 ↓
Answer
```

- with memory
```
    Conversation History
          ↓
    User Question
          ↓
        Prompt
          ↓
        LLM
          ↓
        Answer
```

#### Chat History in LangChain
- LangChain stores conversations as messages.
- Chat message history stores a history of the message interactions in a chat.
- in langchain managing chathistory allows LLM powered applications to remember past interactions and maintain multi turn, context aware conversations. 

In [7]:
# example 
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
import os 
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq
llm = ChatGroq(model=os.getenv("groq_model_name"))

# conversation
history = [
    HumanMessage(content="My name is Subbu"),
    AIMessage(content="Nice to meet you")
]

# new question arrives 
HumanMessage(content="What is my name")

# complete prompt we are sending to llm
[
 HumanMessage("My name is Subbu"),
 AIMessage("Nice to meet you"),
 HumanMessage("What is my name?")
]
# Now the model Answers correctly 

[HumanMessage(content='My name is Subbu', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Nice to meet you', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='What is my name?', additional_kwargs={}, response_metadata={})]

#### ChatTemplate + History

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        ("system","Your helpful"),
        MessagesPlaceholder(
            "chat_history"
        ),
        ("human","{question}")
    ]
)

# now invloke llm to get the reponse 

response = prompt.invoke(
    {
        "chat_history": history,
        "question": "what is my name"
    }
)
print(response)
# here langchain automatically inserts History.
# here all the message types will store here there is a problem with 

messages=[SystemMessage(content='Your helpful', additional_kwargs={}, response_metadata={}), HumanMessage(content='My name is Subbu', additional_kwargs={}, response_metadata={}), AIMessage(content='Nice to meet you', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='what is my name', additional_kwargs={}, response_metadata={})]


- So here before memory Prompt is like ```user: what i my name```
- After memory Prompt is like 
```
User:
My name is Subbu

AI:
Nice to meet you

User:
What is my name?
``` 
- there is a huge difference in before and after  

##### Message Types Stored 
- Human Message : users messages
- AI messages : Asisstance reponses 
- System Messages : Instructions for the llm

#### Main here Problem With Unlimited Memory

- suppose here conversation becomes 500 messages, 1000 messages, 50000 messages.
- sending evrything and evrytime becomes expensive 
- problems
    - More Tokens
    - More cost
    - More latency
    - Context window limits----> so actually for all this problems we need to maintaing smarter memory that is handling by langchain.

#### Different Memory Strategies
- LangChain uses different memory strategies to retain context across LLM interactions.
- These range from short-term memory (thread history) to long-term storage (semantic, episodic, or procedural memory).
- Selecting the right strategy helps prevent context drift and model hallucinations as conversations grow longer.

##### 1. Short-Term (Conversation-Scoped) Memory
- these strategies track ongoing chat sessions with a single converstation threads.

#### ConversationBufferMemory: 
- It stores entire conversation history exactly as it occured verbatiam. This is ideal for short chats, but easily casues models to exceed their context window in longer interactions.
- it is a basic memory implementaion that simply stores the conversation history.
- this Stores entire conversation history in memory without any additional processing.
- Note that additional processing may be required in some situations when the conversation history is too large to fit in the context window of the model.
- it sequentially stores all the interactions (both user input and Ai response) of a chat session in a buffer, and passes this entire history back to the large language model (llm) so that it maintains content and continuity.

###### How its Works 
- when we use the ConversationBufferMemory, langchain logs our dialogs each line by line. Every time we send a new message, the memory grabs all the previous exchanges and injects them into the promot given to the LLM.

###### Implementation Options
- There are two primary ways to utilize it: as a String (concatenated text logs) or as Message Objects (distinct HumanMessage and AIMessage objects). The latter is highly recommended if you are using modern chat models.

- pros :
    * Storing the entire chat log provides the LLM with 100% of the conversation context, so it never forgets details from early in the chat.
    * It is incredibly easy to set up and intuitive to use.

- Cons:
    * Token usage grows linearly. As the conversation gets longer, the prompt sent to the LLM gets larger. This can drastically slow down response times and quickly increase API costs.
    * Token overflow. Eventually, an extensive conversation will exceed the LLM's maximum context window limit, which will crash the app or cause the model to forget information.


In [ ]:
# code  : 
import os 
from dotenv import load_dotenv
load_dotenv()
from langchain_classic.chains import ConversationChain
from langchain_classic.memory import ConversationBufferMemory
from langchain_groq import ChatGroq

llm = ChatGroq(model = os.getenv("groq_model_name"))

memory = ConversationBufferMemory(return_messages = True)
# coversation chain
conversation = ConversationChain(
    llm = llm,
    memory = memory,
    verbose = True # Set to True to see the exact prompts beign sent
)
# interact with the chain
print(conversation.predict(input= "Hi, My name is Alex."))

C:\Users\subramani.v\AppData\Local\Temp\ipykernel_19964\729160864.py:11: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferMemory(return_messages = True)
C:\Users\subramani.v\AppData\Local\Temp\ipykernel_19964\729160864.py:13: LangChainDeprecationWarning: The class `ConversationChain` was deprecated in LangChain 0.2.7 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. Build a conversational agent with `langchain.agents.create_agent` and persist message history via a LangGraph checkpointer.
  conversation = ConversationChain(




> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
[]
Human: Hi, My name is Alex.
AI:

> Finished chain.
Hello Alex, nice to meet you. I'm an artificial intelligence designed to assist and communicate with humans, developed by a team of experts at Meta AI, specifically using a variant of the transformer architecture. We're currently in a conversation environment based on a Markov chain, which allows me to respond to your queries in a more natural and human-like way. How can I help you today, Alex?


In [11]:
print(conversation.predict(input="What is my name?"))



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
[HumanMessage(content='Hi, My name is Alex.', additional_kwargs={}, response_metadata={}), AIMessage(content="Hello Alex, nice to meet you. I'm an artificial intelligence designed to assist and communicate with humans, developed by a team of experts at Meta AI, specifically using a variant of the transformer architecture. We're currently in a conversation environment based on a Markov chain, which allows me to respond to your queries in a more natural and human-like way. How can I help you today, Alex?", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]
Human: What is my name?
AI:

> Finished chain.
Hello Alex, I'm glad you reminded me.

##### ConversationBufferWindowMemory
-  Maintains a rolling window of only the last k interactions. It discards the oldest messages once the limit is reached, ensuring a fixed memory size.
- ConversationBufferWindowMemory keeps a sliding window of the most recent interactions in a conversation. It only uses the last K rounds of dialogue, dropping older messages to save tokens.

##### How it works 
- we define a window size , k.
- each round consist of one human input and one AI response.
- if k =2, the memory only keeps the last 2 exchanges.
- when the third exchange occures, the oldest one is permanently forgotten.

- Pros
    - Predictable costs. Token usage is capped because the prompt size never grows past your window limit
    - Prevents crashes. You will not overflow the LLM's maximum context window during long sessions.
- Cons 
    - Short-term memory loss. The model completely forgets information from earlier in the chat once it slides out of the window.


In [16]:
from langchain_classic.chains import ConversationChain
from langchain_classic.memory import ConversationBufferWindowMemory
from langchain_groq import ChatGroq

# llm details are defined in the above cells so that we can access every time.
memory = ConversationBufferWindowMemory(k =1, return_messages = True)
conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose = True
)

# Round 1
conversation.predict(input = "My favorite colur is blue")
# Round 2 (Model still knows the color because it's within the window of 1)
conversation.predict(input="What is my favorite color?") 
# Round 3 (This forces the color info out of the sliding window)
conversation.predict(input="I also like eating pizza.") 
# Round 4 (Model will no longer remember the color)
conversation.predict(input="What is my favorite color again?")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
[]
Human: My favorite colur is blue
AI:

> Finished chain.


> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
[HumanMessage(content='My favorite colur is blue', additional_kwargs={}, response_metadata={}), AIMessage(content="Blue is often associated with feelings of calmness and tranquility. Did you know that the color blue has been a significant part of human culture for thousands of years? In ancient

"I'm happy to chat with you about your favorite color, but I don't have any information about your personal preferences in my current database. I know we've been discussing pizza, but I don't have any context about your favorite color. If you'd like to tell me, I'd love to hear it and learn more about you!"

In [20]:
# implementation with LCEL 
import os
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.messages import trim_messages
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder


# 2. Configure the Message Trimmer
# This operates exactly like a window memory (k=2 means max 4 total conversational messages)
trimmer = trim_messages(
    max_tokens=4,                  # Tracked limit (corresponds to token count or message count)
    strategy="last",               # Keep the newest messages, drop the oldest ones
    token_counter=len,             # Using 'len' turns this into a pure message-count window
    start_on="human",              # Crucial: Ensures the chain doesn't break by cutting mid-dialogue
    include_system=True,           # Keeps system rules pinned at the top regardless of window size
)

# 3. Design the Prompt Template
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Keep answers brief."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

# 4. Construct the LCEL Production Chain
# The trimmer processes the state before passing data sequentially into the prompt & model
chain = (
    {
        "input": lambda x: x["input"],
        "chat_history": lambda x: trimmer.invoke(x["chat_history"]),
    }
    | prompt
    | llm
)

# --- Simulation: Watch the Window Slide ---

# Mocking a growing chat history database
history = [
    HumanMessage(content="My favorite framework is LangChain."),
    AIMessage(content="Awesome choice!"),
    HumanMessage(content="I live in New York City."),
    AIMessage(content="NYC is a vibrant place!"),
]

# Run interaction 1 (The model will remember NYC because it is within the last 4 messages)
response_1 = chain.invoke({
    "input": "What city do I live in?",
    "chat_history": history
})
print("Response 1:", response_1.content)

# Update our state history with the new exchange
history.extend([
    HumanMessage(content="What city do I live in?"),
    AIMessage(content=response_1.content)
])

# Run interaction 2 (The oldest info—favorite framework—has now slid out of the window)
response_2 = chain.invoke({
    "input": "What is my favorite framework?",
    "chat_history": history
})
print("Response 2:", response_2.content)


Response 1: New York City.
Response 2: I don't have that information.


##### ConversationTokenBufferMemory
- Similar to the window memory, but it removes the oldest messages based on the number of tokens instead of a fixed message count.
- This allows for precise control over context limits.
- ConversationTokenBufferMemory keeps a buffer of recent interactions in memory, slicing old conversations away based on a strict token length rather than the number of dialogue rounds.

##### How It Works
- instead of using a message count k (like window memory),it requires an llm instance to track exact token structures.
- we define a ```max_token_limit```
- it actively drops the oldest messages as soon as the total token count of the history exeeds your limit.
- This approach provides precise control over API billing expenses and protects against model context length failures. 

In [24]:
## code example:
from langchain_classic.memory import ConversationTokenBufferMemory
from langchain_classic.chains import ConversationChain
llm = llm
memory = ConversationTokenBufferMemory(llm=llm, max_token_limit = 60, return_messages = True)

conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose = True
)

# Interaction 1
conversation.predict(input="I am traveling to Tokyo tomorrow morning.")
# Interaction 2 (Still within token limit)
conversation.predict(input="What is my destination city?")
# Interaction 3 (Pushes the first message out of the strict token buffer)
conversation.predict(input="Can you recommend a packing list for five days?")

C:\Users\subramani.v\AppData\Local\Temp\ipykernel_19964\1229398684.py:5: LangChainDeprecationWarning: The class `ConversationTokenBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationTokenBufferMemory(llm=llm, max_token_limit = 60, return_messages = True)




> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
[]
Human: I am traveling to Tokyo tomorrow morning.
AI:


c:\Users\subramani.v\AppData\Local\Programs\Python\Python310\lib\site-packages\langchain_core\language_models\base.py:447: UserWarning: Using fallback GPT-2 tokenizer for token counting. Token counts may be inaccurate for non-GPT-2 models. For accurate counts, use a model-specific method if available.
  return len(self.get_token_ids(text))
c:\Users\subramani.v\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\subramani.v\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either


> Finished chain.


> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
[]
Human: What is my destination city?
AI:

> Finished chain.


> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
[HumanMessage(content='What is my destination city?', additional_kwargs={}, response_metadata={}), AIMessage(content="I don't have that information. Our conversation just started, and I don't have any prior knowledge about our interaction. Could you please tell me your des

"I'd be happy to help you with a packing list for a five-day trip. Since I don't know your destination city yet, I'll provide you with a general list of essentials that can be applied to most travel situations.\n\nFor a five-day trip, you'll want to pack clothing that can be mixed and matched to create multiple outfits. Here's a suggested list of items to get you started:\n\n1. **Clothing:**\n\t* 3-4 tops or shirts (lightweight and quick-drying)\n\t* 2-3 pairs of pants or shorts\n\t* 1-2 dresses or skirts (optional)\n\t* 1-2 light jackets or sweaters (depending on the weather)\n\t* Undergarments and socks\n\t* Comfortable walking shoes\n\t* Sandals or flip-flops\n\t* Dress shoes or nicer shoes (if you plan to go out in the evenings)\n2. **Toiletries:**\n\t* Toothbrush and toothpaste\n\t* Deodorant\n\t* Shampoo and conditioner\n\t* Soap or body wash\n\t* Razor and shaving cream (if applicable)\n\t* Makeup and makeup remover (if applicable)\n\t* Hairbrush or comb\n\t* Contact lenses and 

In [25]:
## 2. Modern Production Equivalent (LCEL + Trimming)
import os
from langchain_core.messages import HumanMessage, AIMessage, trim_messages
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

llm = llm

# Replacement for ConversationTokenBufferMemory
token_trimmer = trim_messages(
    max_tokens=100,               # Enforce a strict max cap of 100 tokens 
    strategy="last",              # Maintain the newest dialog data
    token_counter=llm,          #
    start_on="human",             # Keeps chat validation from failing on tool/AI fragments
    include_system=True           # Keeps structural instructions safe from trimming
)

# Define your workflow prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an AI assistant. Answer short and precisely."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

# Complete LCEL Chain
chain = (
    {
        "input": lambda x: x["input"],
        "chat_history": lambda x: token_trimmer.invoke(x["chat_history"]),
    }
    | prompt
    | llm
)

# Example Usage
history = [
    HumanMessage(content="My private access key code is XYZ-999."),
    AIMessage(content="Got it. Securely noted."),
    HumanMessage(content="I am currently launching a server cluster in Oregon."),
    AIMessage(content="Oregon data centers are ready.")
]

# This execution strips the oldest message block automatically if it breaches 100 tokens
response = chain.invoke({
    "input": "Where am I spinning up infrastructure?",
    "chat_history": history
})
print(response.content)


You're in the AWS Oregon region.


#### Key Variations to Keep Straight
- ConversationBufferWindowMemory: Truncates based on interaction count (e.g., drop anything older than 3 entries).
- ConversationTokenBufferMemory: Truncates based on token weight (e.g., drop anything over 500 tokens).
- ConversationSummaryBufferMemory: Instead of dropping old messages entirely, it condenses them into a summary text chunk to preserve partial context.

### 2. Context-Compression Memory
- these Strategies compress or reduce the size of past conversation history to save tokens without losing critical details.

- *ConversationSummaryMemory* : Uses an LLM to dynamically sonversation so far. A new message comes, then the summary is updated, and only the summarized version is passed to the prompt.
- *ConversationSummaryBufferMemory* : A hybrid of the buffer and summary methods. it retains the verbatiam transcript (captures everything) of the most recent interations in the buffer while summarizing the older messages into a condensed block.

### 3. Entity & Knowledge-Based Memory
- These methods go beyond sequential message storage to retain specific details about people, objects, or concepts.

- *ConversationEntityMemory* :  Automatically extracts information about specific "entities" (e.g., a person's name or a product) from the conversation and continually updates a profile of facts about those entities.
- *Knowledge Graph Memory* : Extracts triplets of information (subject, predicate, object) to build a structured map of concepts and how they relate to one another

### 4. Long-Term & External Memory
- These strategies allow agents to remember facts across different threads or sessions.
- *Vector Store-Backed Memory*: Converts conversation history into vector embeddings and stores them in a vector database (like Chroma, FAISS, or Pinecone). When recalling memories, it runs a semantic search over past inputs and outputs.
- *LangGraph Stores*: Modern orchestration in LangGraph allows for long-term data storage scoping, where systems recall memories outside a single thread by saving them into custom namespaces.

In [27]:
## Coversational Entity Memory

from langchain_classic.chains import ConversationChain
from langchain_classic.memory import ConversationEntityMemory
from langchain_classic.chains.conversation.prompt import ENTITY_MEMORY_CONVERSATION_TEMPLATE

# Initialize the LLM (Required by Entity Memory to extract and summarize entities)
llm = llm

# Initialize ConversationEntityMemory
# We use return_messages=False here for clean string rendering of history
memory = ConversationEntityMemory(llm=llm, return_messages=False)

# Build the Conversation Chain using LangChain's built-in entity template
conversation = ConversationChain(
    llm=llm,
    prompt=ENTITY_MEMORY_CONVERSATION_TEMPLATE,
    memory=memory,
    verbose=True # Set to True to see entity extraction happening live
)

# --- First Interaction ---
print("--- Interaction 1 ---")
response1 = conversation.predict(input="My friend Sarah works as a Data Scientist at Google.")
print(f"AI: {response1}\n")

# Inspect what the memory extracted
memory_variables = memory.load_memory_variables({"input": "tell me about Sarah"})
print("Extracted Entities & Summaries:")
print(memory_variables['entities'])
# Output will look like: {'Sarah': 'Sarah is a friend who works as a Data Scientist at Google.', 'Google': 'Google is where Sarah works as a Data Scientist.'}


# --- Second Interaction (Contextual Verification) ---
print("\n--- Interaction 2 ---")
# The LLM knows who Sarah is because the entity memory injects Sarah's summary into the prompt
response2 = conversation.predict(input="What company does Sarah work for again?")
print(f"AI: {response2}\n")


C:\Users\subramani.v\AppData\Local\Temp\ipykernel_19964\2999718939.py:12: LangChainDeprecationWarning: The class `ConversationEntityMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationEntityMemory(llm=llm, return_messages=False)
c:\Users\subramani.v\AppData\Local\Programs\Python\Python310\lib\site-packages\pydantic\main.py:263: LangChainDeprecationWarning: The class `InMemoryEntityStore` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain

--- Interaction 1 ---


> Entering new ConversationChain chain...
Prompt after formatting:
You are an assistant to a human, powered by a large language model trained by OpenAI.

You are designed to be able to assist with a wide range of tasks, from answering simple questions to providing in-depth explanations and discussions on a wide range of topics. As a language model, you are able to generate human-like text based on the input you receive, allowing you to engage in natural-sounding conversations and provide responses that are coherent and relevant to the topic at hand.

You are constantly learning and improving, and your capabilities are constantly evolving. You are able to process and understand large amounts of text, and can use this knowledge to provide accurate and informative responses to a wide range of questions. You have access to some personalized information provided by the human in the Context section below. Additionally, you are able to generate your own text based on t

In [30]:
### Knowledge Graph Memory
from langchain_classic.chains import ConversationChain
from langchain_classic.memory import ConversationKGMemory
from langchain_classic.prompts import PromptTemplate

# Initialize the LLM
llm = llm


# Initialize Knowledge Graph Memory
memory = ConversationKGMemory(llm=llm, return_messages=False)

# KG Memory requires a prompt template that specifically includes a {history} placeholder
# where the retrieved graph triplets will be injected.
template = """The following is a conversation between a human and an AI. 
The AI is vibrant and provides lots of specific details from its context. 
If the AI does not know the answer to a question, it truthfully says it does not know.

Relevant pieces of previous knowledge:
{history}

Current conversation:
Human: {input}
AI:"""

prompt = PromptTemplate(
    input_variables=["history", "input"], 
    template=template
)

# Build the Conversation Chain
conversation_kg = ConversationChain(
    llm=llm,
    prompt=prompt,
    memory=memory,
    verbose=True
)

# --- First Interaction ---
print("--- Interaction 1 ---")
response1 = conversation_kg.predict(input="Alex is a developer and Alex loves Python.")
print(f"AI: {response1}\n")

# --- Second Interaction ---
print("--- Interaction 2 ---")
response2 = conversation_kg.predict(input="Python belongs to the software domain.")
print(f"AI: {response2}\n")

# --- Querying the structural memory directly ---
print("--- Inspecting the Knowledge Graph Memory ---")

# Look up what the graph knows about 'Alex'
kg_variables = memory.load_memory_variables({"input": "Who is Alex?"})
print("Injected Graph Context:")
print(kg_variables['history'])
# Output will display formatted triplets like: "On Alex: Alex is developer. Alex loves Python."

# You can also programmatically extract raw triplets from a string using the memory tool
triplets = memory.get_knowledge_triplets("Alex is friends with Sam who works at OpenAI")
print("\nExtracted Raw Knowledge Triplets:")
for t in triplets:
    print(f"Subject: {t.subject} | Predicate: {t.predicate} | Object: {t.object_}")


--- Interaction 1 ---


> Entering new ConversationChain chain...
Prompt after formatting:
The following is a conversation between a human and an AI. 
The AI is vibrant and provides lots of specific details from its context. 
If the AI does not know the answer to a question, it truthfully says it does not know.

Relevant pieces of previous knowledge:


Current conversation:
Human: Alex is a developer and Alex loves Python.
AI:

> Finished chain.
AI: That's an interesting combination. As a Python enthusiast, Alex might be drawn to popular Python frameworks such as Django or Flask for web development, or libraries like NumPy and Pandas for data analysis. 

Alex may also be familiar with object-oriented programming principles, which are well-supported in Python. Additionally, they might have heard about the popular Python libraries for automation, such as Scrapy for web scraping or Robot Framework for test automation.

However, I don't have any specific information about Alex's current pr

#### RunnableWithMessageHistory 
- it is the standard class in modern langchain, it is used to attach chat memory to an excecution chain.
- instead of treating memory as global object wrapped around an llm, RunnableWithMessageHistory wraps an entire excecution chaina nd dynamically injects or saves session histories on-the-fly using unique keys like ```session_id```. this design is multi-user and thread-safe out of the box, making it suitable for production systems.


##### How It Works Under the Hood
- 1. *Intercepts Input* : When you invoke the chain, it intercepts the execution context and looks for a unique indentifier(eg: session_id, "user123") in the metadata
- 2. *Fetched History* : it triggers a custom looker function  It triggers a custom lookup function you write to fetch that user's specific history from a database (like Redis, Postgres, or In-Memory stores)
- 3. *Injects Context* :  It feeds those retrieved historical messages directly into a designated MessagesPlaceholder in your prompt template.
- 4. *Appends Responses* : After the LLM replies, it appends both your newest query and the model's new response back to the data store.

In [38]:
# standard production implementation using in memory database dictionary.
import os
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_groq import ChatGroq

llm = ChatGroq(model = os.getenv("groq_model_name"))

# 1.first here setup external database / session here using in-memory dictionary for this problem
session_store = {}

# 2.define a retrival factory function 
# This tells LangChain exactly where to fetch/save histories based on session_id
def get_session_history(session_id:str):
    if session_id not in session_store:
        session_store[session_id] = ChatMessageHistory()
    return session_store[session_id]

# 3.Create base prompt and chain
prompt = ChatPromptTemplate.from_messages([
    ("system","your a helpful Assistant."),
    MessagesPlaceholder(variable_name="my_chat_history"), # this holds the history logs
    ("human","{input_query}") # this holds the newest user input
]) 

chain = prompt | llm

# 4. Wrap the base chain with RunnableMessageHistory
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history = get_session_history,
    input_messages_key = "input_query", # this tells us where to extract the user query
    history_messages_key = "my_chat_history"
)

# excecution 
# User A - First turn
response_a1 = with_message_history.invoke(
    {"input_query": "Hi! My name is Jordan and I live in Paris."},
    config={"configurable": {"session_id": "user_session_abc"}} # Crucial session tracker config
)
print("User A Response 1:", response_a1.content)

User A Response 1: Bonjour Jordan! It's lovely to meet you. Paris, the city of love and light, is a beautiful place to call home. What brings you to this enchanting city? Are you a native Parisian or have you moved here from somewhere else?


In [42]:
# call the function to see the session id 
session_id = get_session_history("user_session_abc")
print(session_id)


Human: Hi! My name is Jordan and I live in Paris.
AI: Bonjour Jordan! It's lovely to meet you. Paris, the city of love and light, is a beautiful place to call home. What brings you to this enchanting city? Are you a native Parisian or have you moved here from somewhere else?


In [43]:
# User B - Interleaved conversation (Completely separate workspace)
response_b1 = with_message_history.invoke(
    {"input_query": "What is the capital of Spain?"},
    config={"configurable": {"session_id": "user_session_xyz"}}
)
print("User B Response 1:", response_b1.content)

User B Response 1: The capital of Spain is Madrid.


In [44]:
# User A - Second turn (Remembers context because session_id matches)
response_a2 = with_message_history.invoke(
    {"input_query": "What is my name and where do I live?"},
    config={"configurable": {"session_id": "user_session_abc"}}
)
print("User A Response 2:", response_a2.content)

User A Response 2: Your name is Jordan, and you live in Paris.


#### Important Configurations
- *input_messages_key*:  If your chain expects a raw dictionary input (e.g., {"input_query": "...", "date": "..."}), you must provide this setting so LangChain knows which exact key contains the user's newest string query. If your chain accepts a single raw string or message array, you can omit this.
- config={"configurable": {"session_id": "..."}}: This is required on every single .invoke(), .stream(), or .batch() execution call to direct tracking states accurately

##### Memory Storage Locations
- in memory ```dict,list```, it is temporary, lost after restart.
- redis : common production choice, stores chat history, fast.
- postgres sql : stores conversations permanently 
- Mongodb: store converstion documents
- vector databses : It stores semantic memories, and it is useful for long term memory. Chroma db , pinecone, FAISS


##### Over View
- Memory in LangChain is the mechanism used to store and manage conversation history so that previous interactions can be included in future prompts, enabling context-aware conversations. LangChain supports multiple memory strategies such as buffer, window, summary, and entity-based memory, and modern implementations commonly use RunnableWithMessageHistory for automatic chat history management.